# Kimi Audio + Pre-existing Transcript Few-Shot AD Detection

## 1. Imports & Configuration

In [ ]:
import json
import tempfile
from pathlib import Path

import torch
import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from kimia_infer.api.kimia import KimiAudio

from llm_utils.config import PROJECT_ROOT, CACHE_DIR
MODEL_ID     = "moonshotai/Kimi-Audio-7B-Instruct"
TEXT_DIR      = PROJECT_ROOT / "data/Text"

LOCAL_MODEL_PATH = snapshot_download(MODEL_ID, cache_dir=CACHE_DIR)
PAIR_NUM = 3

Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

## 2. Model Loading

In [2]:
# Kimi Audio -> GPU 0
model = KimiAudio(model_path=LOCAL_MODEL_PATH, load_detokenizer=True)
print("Kimi Audio model loaded on cuda:0.")

2026-04-03 14:21:20.606 | INFO     | kimia_infer.api.kimia:__init__:16 - Loading kimi-audio main model
2026-04-03 14:21:20.608 | INFO     | kimia_infer.api.kimia:__init__:25 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-03 14:21:20.609 | INFO     | kimia_infer.api.kimia:__init__:26 - Loading whisper model
`torch_dtype` is deprecated! Use `dtype` instead!
using normal flash attention


Loading checkpoint shards:   0%|          | 0/36 [00:00<?, ?it/s]

2026-04-03 14:21:29.620 | INFO     | kimia_infer.api.prompt_manager:__init__:20 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-04-03 14:21:29.622 | INFO     | kimia_infer.api.prompt_manager:__init__:21 - Loading whisper model
2026-04-03 14:21:30.482 | INFO     | kimia_infer.api.prompt_manager:__init__:30 - Loading text tokenizer
2026-04-03 14:21:30.651 | INFO     | kimia_infer.api.kimia:__init__:41 - Loading detokenizer


ninja: no work to do.


/root/autodl-tmp/envs/kimi/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading '/root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b/vocoder/model.pt'
Complete.
using rope base theta = 10000.0, interpolation factor = 1.0
Currently using bfloat16 for PrefixFlowMatchingDetokenizer
Kimi Audio model loaded on cuda:0.


## 3. Utility Functions

In [3]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="kimi_wav_"))

# Mapping from dataset name prefix to transcript subdirectory
TRANSCRIPT_MAP = {
    "Pitt":        "Pitt_Transcript",
    "Pitt-origin": "Pitt_Transcript",
    "Lu":          "Lu_Transcript",
}


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path


def load_transcript(session_id: str, label: str, dataset_prefix: str) -> str:
    """Load pre-existing transcript from Text directory (raw content, no processing)."""
    transcript_subdir = TRANSCRIPT_MAP[dataset_prefix]
    txt_path = TEXT_DIR / transcript_subdir / label / f"{session_id}.txt"
    if not txt_path.exists():
        return ""
    return txt_path.read_text(encoding="utf-8").strip()

## 4. Classification & Prompting

In [4]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech and transcription. "
    "You analyze speech patterns and transcribed text including: word-finding "
    "difficulties, semantic paraphasias, empty speech, reduced syntactic complexity, "
    "repetitions, incomplete utterances, and pragmatic impairments. Based on the "
    "audio and transcription, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample and read the transcription carefully. "
    "Based on both the speech characteristics and the transcription content, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)


def build_example(label: str, wav_path, transcript: str):
    """Build a single few-shot example (text + audio + answer)."""
    hint = "healthy control" if label == "Control" else "dementia"
    return [
        {"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\n" + f"This is a {hint} speaker.\nTranscription: {transcript}"},
        {"role": "user",      "message_type": "audio", "content": str(wav_path)},
        {"role": "assistant", "message_type": "text",  "content": label},
    ]


def classify_audio(wav_path: Path, transcript: str,
                    control_wavs: list, control_transcripts: list,
                    dementia_wavs: list, dementia_transcripts: list) -> str:
    """Classify a single audio file with few-shot examples, using audio + transcript."""
    messages = []
    for i in range(len(control_wavs)):
        messages.extend(build_example("Control", control_wavs[i], control_transcripts[i]))
    for i in range(len(dementia_wavs)):
        messages.extend(build_example("Dementia", dementia_wavs[i], dementia_transcripts[i]))
    # Test sample
    messages.append({"role": "user",      "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT + "\n\nTranscription: " + transcript})
    messages.append({"role": "user",      "message_type": "audio", "content": str(wav_path)})

    _, text = model.generate(messages, output_type="text", max_new_tokens=256)
    return text


def parse_prediction(raw: str) -> str | None:
    """Extract prediction from model output via keyword matching."""
    text = raw.lower()
    has_dementia = "dementia" in text
    has_control  = "control" in text or "healthy" in text
    if has_dementia and not has_control:
        return "Dementia"
    if has_control and not has_dementia:
        return "Control"
    return None

## 5. Evaluation Framework

In [5]:
OUTPUT_DIR = Path("/root/autodl-tmp/Back_to_Origin/ad_detection/train_notebook/LLM/results/kimi_audio_transcript_fewshot_result")


def get_examples(df, audio_dir, dataset_prefix, n=PAIR_NUM):
    """Pick the first n Control and first n Dementia samples as few-shot examples, with pre-existing transcripts."""
    label_map = {0: "Control", 1: "Dementia"}
    audio_dir = Path(audio_dir)
    examples = {}
    for ad_val, label in label_map.items():
        rows = df[df["ad"] == ad_val].iloc[:n]
        wavs, ids, transcripts = [], [], []
        for _, row in rows.iterrows():
            matches = list(audio_dir.glob(f"{label}/{row['session_id']}.*"))
            wav = ensure_wav(matches[0])
            wavs.append(wav)
            ids.append(row["session_id"])
            transcripts.append(load_transcript(row["session_id"], label, dataset_prefix))
        examples[label] = {"session_ids": ids, "wavs": wavs, "transcripts": transcripts}
    print(f"  Few-shot examples: Control={examples['Control']['session_ids']}, Dementia={examples['Dementia']['session_ids']}")
    return examples


def evaluate_dataset(csv_path, audio_dir, dataset_prefix, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    # Select few-shot examples from this dataset
    examples = get_examples(df, audio_dir, dataset_prefix)
    example_ids = set(examples["Control"]["session_ids"] + examples["Dementia"]["session_ids"])
    control_wavs        = examples["Control"]["wavs"]
    control_transcripts = examples["Control"]["transcripts"]
    dementia_wavs        = examples["Dementia"]["wavs"]
    dementia_transcripts = examples["Dementia"]["transcripts"]

    predictions, skipped = [], 0

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        # Skip few-shot example samples
        if row["session_id"] in example_ids:
            skipped += 1
            continue
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            wav = ensure_wav(matches[0])
            transcript = load_transcript(row["session_id"], label_dir, dataset_prefix)
            raw = classify_audio(wav, transcript,
                                 control_wavs, control_transcripts,
                                 dementia_wavs, dementia_transcripts)
            pred = parse_prediction(raw)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            raw, pred = "OOM", None
        except Exception as e:
            raw, pred = str(e), None
        finally:
            torch.cuda.empty_cache()
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    # Save predictions to CSV
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUTPUT_DIR / f"{name}.csv"
    pd.DataFrame(predictions).to_csv(out_csv, index=False)
    print(f"  Saved to {out_csv}")

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred) * 100:.2f}%")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1) * 100:.2f}%")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) * 100:.2f}%")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

## 6. Evaluation on Datasets

### 6.1 Raw Audio

In [6]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-origin-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt-origin", "Pitt-origin", "Pitt-origin_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/Pitt-origin"
evaluate_dataset(csv, audio_dir, "Pitt-origin", "Pitt-origin-raw")

[Pitt-origin-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/Pitt-origin, exists=True
  Few-shot examples: Control=[], Dementia=[]


Pitt-origin-raw:   0%|          | 1/552 [00:01<12:37,  1.38s/it]

  DEBUG [0] session=002-0 raw='Control' pred=Control


Pitt-origin-raw:   0%|          | 2/552 [00:01<07:20,  1.25it/s]

  DEBUG [1] session=002-1 raw='Control' pred=Control


Pitt-origin-raw:   1%|          | 3/552 [00:02<05:34,  1.64it/s]

  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-origin-raw: 100%|██████████| 552/552 [03:29<00:00,  2.64it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Pitt-origin-raw.csv
[Pitt-origin-raw]
  Accuracy:    65.22%
  F1:          0.6350
  Control Acc: 79.42%
  Dementia Acc:54.05%
  Valid: 552/552  Skipped: 0


In [7]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu", "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/raw/Lu, exists=True
  Few-shot examples: Control=[], Dementia=[]


Lu-raw:   1%|▏         | 1/74 [00:00<00:18,  3.90it/s]

  DEBUG [0] session=F22_000 raw='Control' pred=Control


Lu-raw:   3%|▎         | 2/74 [00:00<00:17,  4.01it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-raw:   4%|▍         | 3/74 [00:00<00:19,  3.63it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-raw: 100%|██████████| 74/74 [00:17<00:00,  4.14it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Lu-raw.csv
[Lu-raw]
  Accuracy:    52.70%
  F1:          0.4068
  Control Acc: 75.00%
  Dementia Acc:31.58%
  Valid: 74/74  Skipped: 0


### 6.2 Demucs

In [8]:
csv       = PROJECT_ROOT / "data/processed/Pitt-origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-origin", "Pitt-origin-Demucs")

[Pitt-origin-Demucs] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/denoised/Pitt-origin-Demucs, exists=True
  Few-shot examples: Control=[], Dementia=[]


Pitt-origin-Demucs:   0%|          | 1/552 [00:00<02:43,  3.37it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-origin-Demucs:   0%|          | 2/552 [00:00<02:46,  3.30it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-origin-Demucs:   1%|          | 3/552 [00:00<02:41,  3.40it/s]

  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-origin-Demucs: 100%|██████████| 552/552 [02:44<00:00,  3.35it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Pitt-origin-Demucs.csv
[Pitt-origin-Demucs]
  Accuracy:    66.85%
  F1:          0.7419
  Control Acc: 43.62%
  Dementia Acc:85.11%
  Valid: 552/552  Skipped: 0


In [9]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu", "Lu-Demucs")

[Lu-Demucs] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/denoised/Lu-Demucs, exists=True
  Few-shot examples: Control=[], Dementia=[]


Lu-Demucs:   1%|▏         | 1/74 [00:00<00:20,  3.65it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Demucs:   3%|▎         | 2/74 [00:00<00:17,  4.10it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Demucs:   4%|▍         | 3/74 [00:00<00:18,  3.92it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Demucs: 100%|██████████| 74/74 [00:16<00:00,  4.39it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Lu-Demucs.csv
[Lu-Demucs]
  Accuracy:    60.81%
  F1:          0.6133
  Control Acc: 61.11%
  Dementia Acc:60.53%
  Valid: 74/74  Skipped: 0


### 6.3 Denoiser

In [10]:
csv       = PROJECT_ROOT / "data/processed/Pitt-origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-origin", "Pitt-origin-Denoiser")

[Pitt-origin-Denoiser] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/denoised/Pitt-origin-Denoiser, exists=True
  Few-shot examples: Control=[], Dementia=[]


Pitt-origin-Denoiser:   0%|          | 1/552 [00:00<02:28,  3.70it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-origin-Denoiser:   0%|          | 2/552 [00:00<02:32,  3.60it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-origin-Denoiser:   1%|          | 3/552 [00:00<02:24,  3.80it/s]

  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-origin-Denoiser: 100%|██████████| 552/552 [02:26<00:00,  3.76it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Pitt-origin-Denoiser.csv
[Pitt-origin-Denoiser]
  Accuracy:    62.14%
  F1:          0.7296
  Control Acc: 25.10%
  Dementia Acc:91.26%
  Valid: 552/552  Skipped: 0


In [11]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu", "Lu-Denoiser")

[Lu-Denoiser] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/denoised/Lu-Denoiser, exists=True
  Few-shot examples: Control=[], Dementia=[]


Lu-Denoiser:   1%|▏         | 1/74 [00:00<00:16,  4.40it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Denoiser:   3%|▎         | 2/74 [00:00<00:15,  4.62it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Denoiser:   4%|▍         | 3/74 [00:00<00:15,  4.48it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Denoiser: 100%|██████████| 74/74 [00:15<00:00,  4.76it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Lu-Denoiser.csv
[Lu-Denoiser]
  Accuracy:    55.41%
  F1:          0.6526
  Control Acc: 27.78%
  Dementia Acc:81.58%
  Valid: 74/74  Skipped: 0


### 6.4 FRCRN_SE

In [12]:
csv       = PROJECT_ROOT / "data/processed/Pitt-origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-origin", "Pitt-origin-FRCRN_SE")

[Pitt-origin-FRCRN_SE] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/denoised/Pitt-origin-FRCRN_SE, exists=True
  Few-shot examples: Control=[], Dementia=[]


Pitt-origin-FRCRN_SE:   0%|          | 1/552 [00:00<02:22,  3.87it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control


Pitt-origin-FRCRN_SE:   0%|          | 2/552 [00:00<02:28,  3.71it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-origin-FRCRN_SE:   1%|          | 3/552 [00:00<02:22,  3.85it/s]

  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-origin-FRCRN_SE: 100%|██████████| 552/552 [02:25<00:00,  3.79it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Pitt-origin-FRCRN_SE.csv
[Pitt-origin-FRCRN_SE]
  Accuracy:    66.12%
  F1:          0.7246
  Control Acc: 48.97%
  Dementia Acc:79.61%
  Valid: 552/552  Skipped: 0


In [13]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu", "Lu-FRCRN_SE")

[Lu-FRCRN_SE] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/denoised/Lu-FRCRN_SE, exists=True
  Few-shot examples: Control=[], Dementia=[]


Lu-FRCRN_SE:   1%|▏         | 1/74 [00:00<00:17,  4.16it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-FRCRN_SE:   3%|▎         | 2/74 [00:00<00:16,  4.48it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-FRCRN_SE:   4%|▍         | 3/74 [00:00<00:16,  4.41it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-FRCRN_SE: 100%|██████████| 74/74 [00:14<00:00,  4.94it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Lu-FRCRN_SE.csv
[Lu-FRCRN_SE]
  Accuracy:    60.81%
  F1:          0.6234
  Control Acc: 58.33%
  Dementia Acc:63.16%
  Valid: 74/74  Skipped: 0


### 6.5 MossFormer

In [14]:
csv       = PROJECT_ROOT / "data/processed/Pitt-origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-origin", "Pitt-origin-MossFormer")

[Pitt-origin-MossFormer] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/denoised/Pitt-origin-MossFormer, exists=True
  Few-shot examples: Control=[], Dementia=[]


Pitt-origin-MossFormer:   0%|          | 1/552 [00:00<02:16,  4.03it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control


Pitt-origin-MossFormer:   0%|          | 2/552 [00:00<02:24,  3.81it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-origin-MossFormer:   1%|          | 3/552 [00:00<02:20,  3.91it/s]

  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-origin-MossFormer: 100%|██████████| 552/552 [02:24<00:00,  3.81it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Pitt-origin-MossFormer.csv
[Pitt-origin-MossFormer]
  Accuracy:    66.67%
  F1:          0.7473
  Control Acc: 39.51%
  Dementia Acc:88.03%
  Valid: 552/552  Skipped: 0


In [15]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu", "Lu-MossFormer")

[Lu-MossFormer] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/denoised/Lu-MossFormer, exists=True
  Few-shot examples: Control=[], Dementia=[]


Lu-MossFormer:   1%|▏         | 1/74 [00:00<00:16,  4.33it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-MossFormer:   3%|▎         | 2/74 [00:00<00:15,  4.64it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-MossFormer:   4%|▍         | 3/74 [00:00<00:15,  4.51it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-MossFormer: 100%|██████████| 74/74 [00:16<00:00,  4.50it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Lu-MossFormer.csv
[Lu-MossFormer]
  Accuracy:    60.81%
  F1:          0.6813
  Control Acc: 38.89%
  Dementia Acc:81.58%
  Valid: 74/74  Skipped: 0


### 6.6 Resemble

In [16]:
csv       = PROJECT_ROOT / "data/processed/Pitt-origin-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-origin-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-origin", "Pitt-origin-Resemble")

[Pitt-origin-Resemble] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/denoised/Pitt-origin-Resemble, exists=True
  Few-shot examples: Control=[], Dementia=[]


Pitt-origin-Resemble:   0%|          | 1/552 [00:00<02:32,  3.62it/s]

  DEBUG [0] session=002-0 raw='Control' pred=Control


Pitt-origin-Resemble:   0%|          | 2/552 [00:00<02:40,  3.42it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-origin-Resemble:   1%|          | 3/552 [00:00<02:37,  3.49it/s]

  DEBUG [2] session=002-2 raw='Control' pred=Control


Pitt-origin-Resemble: 100%|██████████| 552/552 [02:45<00:00,  3.34it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Pitt-origin-Resemble.csv
[Pitt-origin-Resemble]
  Accuracy:    65.94%
  F1:          0.7439
  Control Acc: 37.45%
  Dementia Acc:88.35%
  Valid: 552/552  Skipped: 0


In [17]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu", "Lu-Resemble")

[Lu-Resemble] audio_dir=/root/autodl-tmp/Back_to_Origin/ad_detection/data/denoised/Lu-Resemble, exists=True
  Few-shot examples: Control=[], Dementia=[]


Lu-Resemble:   1%|▏         | 1/74 [00:00<00:19,  3.79it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Resemble:   3%|▎         | 2/74 [00:00<00:17,  4.13it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Resemble:   4%|▍         | 3/74 [00:00<00:17,  3.97it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Resemble: 100%|██████████| 74/74 [00:17<00:00,  4.27it/s]

  Saved to /root/autodl-tmp/Back_to_Origin/LLM/kimi_audio_transcript_fewshot_result/Lu-Resemble.csv
[Lu-Resemble]
  Accuracy:    63.51%
  F1:          0.7097
  Control Acc: 38.89%
  Dementia Acc:86.84%
  Valid: 74/74  Skipped: 0
